# 09012024 
## 1. How to debug pipeline
After Ru's instruction and learning about sos workflow, my debugging step are as follow:
* Encounter an error and speculate the possible causes.
* Isolate the problematic code and paste them to a .ipynb (setting the code cell environment as 'sos')
* Adjust/simplify its format and replace placeholders with real path or variables.
* Modify the code according to the output within the Jupyter notebook.



Take the error in [rsem_call_4, rnaseqc_call_4] for example:

* I got an error: variable varVCFfile not defined, possible reasons: 1. varVCFfile not defined in sos workflow(either [global] or [specific_step]); 2. sos variable is not recognizable in specific language environments, e.g. R, python, bash.
* the problematic code:

*R: container=container, expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', entrypoint=entrypoint
    readPicard.duplicate_metrics <- function(source) {*

      stopifnot(length(source) == 1)
      stopifnot(file.exists(source))
      isDir <- file.info(source)$isdir
      if (isDir) {
        files <- system(paste("find -L ", source, " -name \"*.Aligned.sortedByCoord.out{"_wasp_qc" if varVCFfile else ""}.md.metrics\""), intern=TRUE)
        stopifnot(length(files) > 0)
        samples <- gsub(".Aligned.sortedByCoord.out{"_wasp_qc" if varVCFfile else ""}.md.metrics", "", basename(files), fixed=TRUE)
      } else {
        files <- source
        samples <- gsub(".Aligned.sortedByCoord.out{"_wasp_qc" if varVCFfile else ""}.md.metrics", "", basename(files), fixed=TRUE)
      }*

* The issues are: 1. f'{"_wasp_qc" if varVCFfile else ""} is python-like script and not work in R 2.need to add "${}" around varVCFfile to make it a variable in R code block
  
* modified code is listed below:      
*R: container=container, expand= "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', entrypoint=entrypoint
    readPicard.duplicate_metrics <- function(source) {*

      stopifnot(length(source) == 1)
      stopifnot(file.exists(source))
      pattern_suffix <- ifelse("${varVCFfile}"=="","_wasp_qc","")
      pattern <- paste0('*.Aligned.sortedByCoord.out',pattern_suffix,'.md.metrics')
      subtitute <- paste0('.Aligned.sortedByCoord.out',pattern_suffix,'.md.metrics')
      isDir <- file.info(source)$isdir
      if (isDir) {
        files <- system(paste("find -L", source, "-name",pattern), intern=TRUE)
        stopifnot(length(files) > 0)
        samples <- gsub(substitute, "", basename(files), fixed=TRUE)
      } else {
        files <- source
        samples <- gsub(substitutee, "", basename(files), fixed=TRUE)
      }`